In [ ]:
import sys
import os
import torch
import torch.nn as nn

sys.path.append(os.path.abspath('..'))

from data_prep import create_dataloaders
from utils import count_parameters, train_model, evaluate_model
from models.sota_models import get_mobilenet_v2, get_squeezenet

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware accelerator: {device}")

train_loader, val_loader, test_loader, classes, dataset_mean, dataset_std = create_dataloaders(
    dataset_dir="../data/EuroSAT_RGB"
)
num_classes = len(classes)

print(f"Loaded {num_classes} classes: {classes}")
print(f"Calculated Dataset Mean: {dataset_mean}")
print(f"Calculated Dataset Std: {dataset_std}")

Hardware accelerator: cpu
Loaded 10 classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Calculated Dataset Mean: tensor([0.3438, 0.3800, 0.4077])
Calculated Dataset Std: tensor([0.2026, 0.1370, 0.1156])


In [7]:
models_to_train = {
    "MobileNetV2:": get_mobilenet_v2(num_classes=num_classes).to(device),
    "SqueezeNet:": get_squeezenet(num_classes=num_classes).to(device)
}

for name, model in models_to_train.items():
    print(f"{name} Trainable Parameters: {count_parameters(model):,}")

MobileNetV2: Trainable Parameters: 12,810
SqueezeNet: Trainable Parameters: 5,130


In [8]:
results_dict = {}
os.makedirs("../checkpoints", exist_ok=True)

for name, model in models_to_train.items():
    print(f"\n=========================================")
    print(f"         Training {name}                 ")
    print(f"=========================================\n")
    
    criterion = nn.CrossEntropyLoss()
    
    # Filter the optimizer to ONLY update the unfrozen classification layer
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(trainable_params, lr=0.001)
    
    # Run the training loop using your team's shared function
    history = train_model(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        criterion=criterion, 
        optimizer=optimizer, 
        device=device, 
        num_epochs=20
    )
    
    # Save the trained weights
    ckpt_path = f"../checkpoints/{name}.pt"
    torch.save(model.state_dict(), ckpt_path)
    
    # Evaluate and store results
    results = evaluate_model(model, test_loader, device)
    results_dict[name] = results


         Training MobileNetV2:                 

Epoch [01/20] Train Loss: 0.9217 | Train Acc: 0.7570 | Val Loss: 0.5845 | Val Acc: 0.8504 | Time: 49.73s
Epoch [02/20] Train Loss: 0.5888 | Train Acc: 0.8376 | Val Loss: 0.5006 | Val Acc: 0.8620 | Time: 35.25s
Epoch [03/20] Train Loss: 0.5311 | Train Acc: 0.8448 | Val Loss: 0.4812 | Val Acc: 0.8649 | Time: 37.62s
Epoch [04/20] Train Loss: 0.5007 | Train Acc: 0.8492 | Val Loss: 0.4418 | Val Acc: 0.8728 | Time: 35.49s
Epoch [05/20] Train Loss: 0.4787 | Train Acc: 0.8560 | Val Loss: 0.4369 | Val Acc: 0.8783 | Time: 35.54s
Epoch [06/20] Train Loss: 0.4609 | Train Acc: 0.8585 | Val Loss: 0.4550 | Val Acc: 0.8714 | Time: 35.65s
Epoch [07/20] Train Loss: 0.4568 | Train Acc: 0.8601 | Val Loss: 0.4290 | Val Acc: 0.8746 | Time: 35.94s
Epoch [08/20] Train Loss: 0.4580 | Train Acc: 0.8583 | Val Loss: 0.4231 | Val Acc: 0.8800 | Time: 35.41s
Epoch [09/20] Train Loss: 0.4506 | Train Acc: 0.8611 | Val Loss: 0.4174 | Val Acc: 0.8770 | Time: 35.40s
Epoch

In [9]:
print("\n--- FINAL COMPARISON METRICS ---")
for name in models_to_train.keys():
    ckpt_path = f"../checkpoints/{name}.pt"
    size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    acc = results_dict[name]['accuracy'] * 100
    
    print(f"{name}:")
    print(f"  - Test Accuracy : {acc:.2f}%")
    print(f"  - Model Size    : {size_mb:.2f} MB")


--- FINAL COMPARISON METRICS ---
MobileNetV2::
  - Test Accuracy : 86.99%
  - Model Size    : 8.76 MB
SqueezeNet::
  - Test Accuracy : 92.30%
  - Model Size    : 2.84 MB
